# Collecte des données (suite) #
>
>## IV - Données météorologiques ##
>>
>>Après avoir récupéré les données de production, les données sur la position du soleil et les données décrivant l'athmosphère, on cherche maintenant à récupérer des **informations météorologiques** complémentaires, telles que :
>>
>>- la *température* au sol (influe sur la performance des panneaux solaires) ;
>>- la *vitesse du vent* au sol (influe sur la performance des panneaux solaires) ;
>>- l'*humidité* (influe sur la performance des panneaux solaires) ;
>>- la *nébulosité* (influe sur la lumière reçue par les panneaux solaires).
>>
>>On peut trouver ces données dans un dataset mis à disposition par la NASA via une API : **POWER Hourly API** (https://power.larc.nasa.gov/), accessible par le site https://power.larc.nasa.gov/api/pages/#/Data%20Requests/single_point_data_request_api_temporal_hourly_point_get.
>>
>>Cette base de données a une résolution temporelle d'une heure : comme nos données ont une résolution de 30 minutes, nous devrons les interpoler.
>>
>>*Note : une autre base de données qui aurait pu servir pour la température et le vent est la base cams-global-reanalysis-eac4, mais la résolution temporelle de celle ci est de 3h.*

>> ### A - Récupération des données précédentes ###
>>>
>>>Nous commencons par récupérer les données aggrégées à l'étape précédente depuis le Drive du projet :


In [2]:
# On se donne accès au Drive qui contient les fichiers sources
from google.colab import drive

# Monter Google Drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# Chemin pour récupérer le dataset de données athmosphériques
chemin_dataset_cams = '/content/drive/MyDrive/Flux/00_Collecte_datasets/03_Athmosphere/' + "cams_2020_2024.csv"

# Chemin vers le répertoire de travail pour les datasets de données météo
chemin_datasets_meteo = '/content/drive/MyDrive/Flux/00_Collecte_datasets/04_Meteorologie/'

>>>On importe les librairies nécessaires pour manipuler nos données :

In [4]:
# On importe les librairies dont on a besoin pour traiter les datasets
import pandas as pd
import numpy as np

>>>On charge le jeu de données de l'étape précédente :

In [5]:
# On récupère le dataset contenant les données de production, les données astronomiques et les données athmosphériques
df_cams = pd.read_csv(chemin_dataset_cams, parse_dates=['datetime_utc'])
display(df_cams.head())
df_cams.dtypes

/tmp/ipython-input-555239890.py:2: DtypeWarning: Columns (6,7) have mixed types. Specify dtype option on import or set low_memory=False.
  df_cams = pd.read_csv(chemin_dataset_cams, parse_dates=['datetime_utc'])


,datetime_utc,Périmètre,Nature,Consommation,Solaire,Ech. physiques,Stockage batterie,Déstockage batterie,TCO Solaire (%),TCH Solaire (%),...,EYG_TOA,EYG_Clear sky GHI,EYG_Clear sky BHI,EYG_Clear sky DHI,EYG_Clear sky BNI,EYG_GHI,EYG_BHI,EYG_DHI,EYG_BNI,EYG_Reliability
0,2019-12-31 23:00:00+00:00,PACA,Données définitives,6123.0,0.0,3332.0,-,-,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1,2019-12-31 23:30:00+00:00,PACA,Données définitives,5907.0,0.0,2837.0,-,-,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
2,2020-01-01 00:00:00+00:00,PACA,Données définitives,5724.0,0.0,2668.0,-,-,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3,2020-01-01 00:30:00+00:00,PACA,Données définitives,5749.0,0.0,2754.0,-,-,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
4,2020-01-01 01:00:00+00:00,PACA,Données définitives,5700.0,0.0,2886.0,-,-,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


,0
datetime_utc,"datetime64[ns, UTC]"
Périmètre,object
Nature,object
Consommation,float64
Solaire,float64
...,...
EYG_GHI,float64
EYG_BHI,float64
EYG_DHI,float64
EYG_BNI,float64


>>### B - Lieux retenus ###
>>>
>>>On récupère les coordonnées des cinq points significatifs que l'on a précédemment déterminé.
>>>
>>>Pour faciliter l'exploitation ultérieure des données géographiques, on remplace le nom de villes sélectionnées par un code sur trois lettres :    
>>>
>>> - Cruis est encodé par 'CRU' ;
>>> - Saint-Étienne-le-Laus par 'SEL' ;
>>> - Saint-Vallier-de-Thiey par 'SVT' ;
>>> - Bras par 'BRA' ;
>>> - Eygalières par 'EYG'.

In [6]:
# On a retenu 5 lieux pertinents pour l'étude de production d'énergie solaire
# On stocke leur code ainsi que leur coordonnées géographiques (latitude et longitude) dans un dictionnaire
lieux_retenus = {'CRU' : {'latitude': 44.0845, 'longitude': 5.8397},
                 'SEL' : {'latitude': 44.5075, 'longitude': 6.1616},
                 'SVT' : {'latitude': 43.6994, 'longitude': 6.8516},
                 'BRA' : {'latitude': 43.4723, 'longitude': 5.9558},
                 'EYG' : {'latitude': 43.7638, 'longitude': 4.9554}}

>>## C - Collecte des données météorologiques ##
>>>
>>>Le jeu de données **POWER Hourly API** est accessible par le site https://power.larc.nasa.gov/api/pages/#/Data%20Requests/.

>>>Pour faciliter la collecte des données pour chacun de nos points d'intérêts et à l'instar de ce que nous avions fait pour la collecte des données CAMS, on crée une fonction qui interroge l'API de la NASA en fonction des coordonnées géographiques d'un lieu (latitude et longitude) et des dates de début et de fin de la période à observer.
>>>
>>>Un jeu de données est alors enregistré au niveau d'un chemin transmis à la fonction.

In [7]:
def retrieve_nasa (latitude, longitude, date_debut, date_fin, base_chemin) :
  # On formatte nos dates pour qu'elles aient la forme voulue par l'API (yyyymmdd)
  debut = "".join([n for n in date_debut if n != '-'])
  fin = "".join([n for n in date_fin if n != '-'])

  # On construit l'URL de la requete à l'API
  url = "https://power.larc.nasa.gov/api/temporal/hourly/point?start=" + str(debut) + '&end=' + str(fin)
  url += '&latitude=' + str(latitude)+'&longitude=' + str(longitude) + "&community=re"
  url += "&parameters=T2M,WS2M,CLOUD_AMT,RH2M"
  url += "&format=csv&units=metric&header=true&time-standard=utc"

  # T2M : Temperature at 2 Meters
  # RH2M : Relative Humidity at 2 Meters
  # WS2M : Wind Speed at 2 Meters
  # ALLSKY_SFC_LW_DWN : All Sky Surface Longwave Downward Irradiance

  # On lit directement le CSV depuis l'API
  df = pd.read_csv(url, skiprows=12)

  # Créer datetime correct en UTC (NASA POWER fournit YEAR, MO, DY, HR)
  df["datetime_utc"] = pd.to_datetime({"year":  df["YEAR"],
                                       "month": df["MO"],
                                       "day":   df["DY"],
                                       "hour":  df["HR"]},
                                      utc=True)

  # Renommer les colonnes en français
  df = df.rename(columns={'T2M': 'Temperature',
                          'WS2M': 'Vitesse_Vent',
                          'CLOUD_AMT': 'Nebulosite',
                          'RH2M' : "Humidite"})

  # Définir datetime_utc comme index
  df = df.set_index("datetime_utc").sort_index()

  # Remplacer les valeurs manquantes (-999) par np.nan pour qu'on puisse les identifier
  df = df.replace(-999.00, np.nan)

  # Rééchantillonner toutes les 30 minutes et interpoler
  df = df.drop(columns=['YEAR', 'MO', 'DY', 'HR']).resample("30min").interpolate(numeric_only=True)
  df = df.reset_index()

  # On enregistre le dataset
  target = base_chemin + "_" + date_debut + "_" + date_fin + ".csv"
  df.to_csv(target, index=False)

>>>On détermine, à partir du jeu de données précédemment collecté, la date de début et la date de fin de la période couverte.

In [8]:
# On détermine la plage de dates à requêter
# En heures UTC le dataset de départ commence le 31 décembre 2019 et se termine le 31 décembre 2024

date_debut = df_cams.loc[0,'datetime_utc'].strftime('%Y-%m-%d')
date_fin = df_cams.loc[df_cams.shape[0]-1,'datetime_utc'].strftime('%Y-%m-%d')

print("Date de début :", date_debut)
print("Date de fin :", date_fin)

Date de début : 2019-12-31
Date de fin : 2024-12-31


>>>On interroge l'API pour collecter les données athmosphérique pour chaque point d'intérêt :

In [9]:
# On interroge l'API pour récupérer les données NASA de chaque ville
# Pour chaque ville
for ville, location in lieux_retenus.items() :
  # On affiche où on en est rendu dans les villes
  print(ville + "...")

  # Le chemin de base du dataset temporaire sera :
  base_chemin = chemin_datasets_meteo + "NASA_" + ville # La fonction de récupération ajoutera les dates et l'extension

  # Envoi de la requete
  retrieve_nasa(location['latitude'], location['longitude'],
                date_debut, date_fin,
                base_chemin)

  # On indique que la requete est terminée
  print("Ok pour", ville)


CRU...
Ok pour CRU
SEL...
Ok pour SEL
SVT...
Ok pour SVT
BRA...
Ok pour BRA
EYG...
Ok pour EYG


>>### D - Aggrégation des nouvelles données collectées ###
>>>
>>>Les données collectées se présentent pour le moment sous la forme de fichiers csv enregistrés dans le Drive du projet, à raison d'un fichier par point d'intérêt.
>>>
>>>Pour aggréger les nouvelles variables explicatives, on va :
>>>
>>>- Charger les fichiers sous forme de DataFrame Pandas ;
>>>- Renommer les variables en les faisant commencer par le sigle du point d'intérêt ;
>>>-Fusionner les jeux de données sur la base de la variable temporelle.


>>>On charge sous forme de DataFrame Pandas les jeux de données qu'on vient de collecter :

In [10]:
# Chemin des datasets de données NASA
chemin_nasa_CRU = chemin_datasets_meteo + 'NASA_CRU_2019-12-31_2024-12-31.csv'
chemin_nasa_SEL = chemin_datasets_meteo + 'NASA_SEL_2019-12-31_2024-12-31.csv'
chemin_nasa_SVT = chemin_datasets_meteo + 'NASA_SVT_2019-12-31_2024-12-31.csv'
chemin_nasa_BRA = chemin_datasets_meteo + 'NASA_BRA_2019-12-31_2024-12-31.csv'
chemin_nasa_EYG = chemin_datasets_meteo + 'NASA_EYG_2019-12-31_2024-12-31.csv'

In [11]:
# On charge les datasets de la NASA
print("Cruis...")
nasa_CRU = pd.read_csv(chemin_nasa_CRU, parse_dates=['datetime_utc'])

print('Saint-Étienne-le-Laus...')
nasa_SEL = pd.read_csv(chemin_nasa_SEL, parse_dates=['datetime_utc'])

print('Saint-Vallier-de-Thiey...')
nasa_SVT = pd.read_csv(chemin_nasa_SVT, parse_dates=['datetime_utc'])

print('Bras...')
nasa_BRA = pd.read_csv(chemin_nasa_BRA, parse_dates=['datetime_utc'])

print("Eygalières...")
nasa_EYG = pd.read_csv(chemin_nasa_EYG, parse_dates=['datetime_utc'])

print("OK")

Cruis...
Saint-Étienne-le-Laus...
Saint-Vallier-de-Thiey...
Bras...
Eygalières...
OK


>>>Pour parcourir plus facilement les datasets on utilise un dictionnaires ayant pour clé le sigle correspondant au point d'intérêt du dataset :

In [12]:
all_nasa = {'CRU' : nasa_CRU,
            'SEL' : nasa_SEL,
            'SVT' : nasa_SVT,
            'BRA' : nasa_BRA,
            'EYG' : nasa_EYG}

for df in all_nasa.values():
  display(df.head())

,datetime_utc,Temperature,Vitesse_Vent,Nebulosite,Humidite
0,2019-12-31 00:00:00+00:00,2.370,1.030,0.730,81.03
1,2019-12-31 00:30:00+00:00,2.470,1.065,0.885,80.24
2,2019-12-31 01:00:00+00:00,2.570,1.100,1.040,79.45
3,2019-12-31 01:30:00+00:00,2.575,1.110,1.980,79.51
4,2019-12-31 02:00:00+00:00,2.580,1.120,2.920,79.57


,datetime_utc,Temperature,Vitesse_Vent,Nebulosite,Humidite
0,2019-12-31 00:00:00+00:00,-0.400,0.980,83.92,53.66
1,2019-12-31 00:30:00+00:00,-0.450,1.015,42.72,52.82
2,2019-12-31 01:00:00+00:00,-0.500,1.050,1.52,51.98
3,2019-12-31 01:30:00+00:00,-0.525,1.060,36.15,51.13
4,2019-12-31 02:00:00+00:00,-0.550,1.070,70.78,50.28


,datetime_utc,Temperature,Vitesse_Vent,Nebulosite,Humidite
0,2019-12-31 00:00:00+00:00,7.630,1.69,11.510,71.330
1,2019-12-31 00:30:00+00:00,7.955,1.67,9.615,69.335
2,2019-12-31 01:00:00+00:00,8.280,1.65,7.720,67.340
3,2019-12-31 01:30:00+00:00,8.400,1.60,9.250,65.995
4,2019-12-31 02:00:00+00:00,8.520,1.55,10.780,64.650


,datetime_utc,Temperature,Vitesse_Vent,Nebulosite,Humidite
0,2019-12-31 00:00:00+00:00,3.64,1.530,3.57,83.67
1,2019-12-31 00:30:00+00:00,3.92,1.505,3.04,80.61
2,2019-12-31 01:00:00+00:00,4.20,1.480,2.51,77.55
3,2019-12-31 01:30:00+00:00,4.44,1.420,3.04,74.78
4,2019-12-31 02:00:00+00:00,4.68,1.360,3.57,72.01


,datetime_utc,Temperature,Vitesse_Vent,Nebulosite,Humidite
0,2019-12-31 00:00:00+00:00,4.950,0.990,26.290,96.120
1,2019-12-31 00:30:00+00:00,5.060,1.000,24.855,94.295
2,2019-12-31 01:00:00+00:00,5.170,1.010,23.420,92.470
3,2019-12-31 01:30:00+00:00,5.255,0.985,28.050,91.015
4,2019-12-31 02:00:00+00:00,5.340,0.960,32.680,89.560


>>>On renomme les variables des datasets (à l'exeption de la variable `datetime_utc` qui servira à la fusion) en les faisants débuter par le sigle du point d'intérêt correspondant :

In [13]:
# On renomme les colonnes des différents datasets pour qu'elles comprennent le nom de la ville
for ville, df in all_nasa.items() :
    mon_dico = {'Temperature' : ville + '_Temperature',
                'Vitesse_Vent' : ville + '_Vitesse_Vent',
                'Nebulosite' : ville +'_Nebulosite',
                'Humidite' : ville + '_Humidite'}
    df.rename(mon_dico, axis=1, inplace=True)

for df in all_nasa.values():
  display(df.head())

,datetime_utc,CRU_Temperature,CRU_Vitesse_Vent,CRU_Nebulosite,CRU_Humidite
0,2019-12-31 00:00:00+00:00,2.370,1.030,0.730,81.03
1,2019-12-31 00:30:00+00:00,2.470,1.065,0.885,80.24
2,2019-12-31 01:00:00+00:00,2.570,1.100,1.040,79.45
3,2019-12-31 01:30:00+00:00,2.575,1.110,1.980,79.51
4,2019-12-31 02:00:00+00:00,2.580,1.120,2.920,79.57


,datetime_utc,SEL_Temperature,SEL_Vitesse_Vent,SEL_Nebulosite,SEL_Humidite
0,2019-12-31 00:00:00+00:00,-0.400,0.980,83.92,53.66
1,2019-12-31 00:30:00+00:00,-0.450,1.015,42.72,52.82
2,2019-12-31 01:00:00+00:00,-0.500,1.050,1.52,51.98
3,2019-12-31 01:30:00+00:00,-0.525,1.060,36.15,51.13
4,2019-12-31 02:00:00+00:00,-0.550,1.070,70.78,50.28


,datetime_utc,SVT_Temperature,SVT_Vitesse_Vent,SVT_Nebulosite,SVT_Humidite
0,2019-12-31 00:00:00+00:00,7.630,1.69,11.510,71.330
1,2019-12-31 00:30:00+00:00,7.955,1.67,9.615,69.335
2,2019-12-31 01:00:00+00:00,8.280,1.65,7.720,67.340
3,2019-12-31 01:30:00+00:00,8.400,1.60,9.250,65.995
4,2019-12-31 02:00:00+00:00,8.520,1.55,10.780,64.650


,datetime_utc,BRA_Temperature,BRA_Vitesse_Vent,BRA_Nebulosite,BRA_Humidite
0,2019-12-31 00:00:00+00:00,3.64,1.530,3.57,83.67
1,2019-12-31 00:30:00+00:00,3.92,1.505,3.04,80.61
2,2019-12-31 01:00:00+00:00,4.20,1.480,2.51,77.55
3,2019-12-31 01:30:00+00:00,4.44,1.420,3.04,74.78
4,2019-12-31 02:00:00+00:00,4.68,1.360,3.57,72.01


,datetime_utc,EYG_Temperature,EYG_Vitesse_Vent,EYG_Nebulosite,EYG_Humidite
0,2019-12-31 00:00:00+00:00,4.950,0.990,26.290,96.120
1,2019-12-31 00:30:00+00:00,5.060,1.000,24.855,94.295
2,2019-12-31 01:00:00+00:00,5.170,1.010,23.420,92.470
3,2019-12-31 01:30:00+00:00,5.255,0.985,28.050,91.015
4,2019-12-31 02:00:00+00:00,5.340,0.960,32.680,89.560


>>>On fusionne les datasets en se basant sur la variable `datetime_utc` qui contient la date et l'heure des observations :  

In [14]:
# On va merger sur les colonnes 'datetime_utc' en n'ajoutant aucune ligne aux données cams
for df in all_nasa.values():
  df_cams = df_cams.merge(right=df, on='datetime_utc', how='left')

>>>On affiche le début du jeu de données pour avoir une première visualisation des nouvelles données :

In [15]:
# On affiche les premières lignes du dataset consolidé
display(df_cams.head())
print("Taille du dataset :", df_cams.shape)

,datetime_utc,Périmètre,Nature,Consommation,Solaire,Ech. physiques,Stockage batterie,Déstockage batterie,TCO Solaire (%),TCH Solaire (%),...,SVT_Nebulosite,SVT_Humidite,BRA_Temperature,BRA_Vitesse_Vent,BRA_Nebulosite,BRA_Humidite,EYG_Temperature,EYG_Vitesse_Vent,EYG_Nebulosite,EYG_Humidite
0,2019-12-31 23:00:00+00:00,PACA,Données définitives,6123.0,0.0,3332.0,-,-,0.0,0.0,...,29.440,57.600,3.230,1.620,5.700,86.880,2.720,1.220,51.750,100.000
1,2019-12-31 23:30:00+00:00,PACA,Données définitives,5907.0,0.0,2837.0,-,-,0.0,0.0,...,24.785,56.165,3.345,1.565,4.635,84.455,2.760,1.405,54.215,98.790
2,2020-01-01 00:00:00+00:00,PACA,Données définitives,5724.0,0.0,2668.0,-,-,0.0,0.0,...,20.130,54.730,3.460,1.510,3.570,82.030,2.800,1.590,56.680,97.580
3,2020-01-01 00:30:00+00:00,PACA,Données définitives,5749.0,0.0,2754.0,-,-,0.0,0.0,...,10.340,54.250,3.650,1.435,3.040,79.380,2.675,1.635,33.325,96.065
4,2020-01-01 01:00:00+00:00,PACA,Données définitives,5700.0,0.0,2886.0,-,-,0.0,0.0,...,0.550,53.770,3.840,1.360,2.510,76.730,2.550,1.680,9.970,94.550


Taille du dataset : (87696, 90)


>>### E - Enregistrement des données NASA collectées ###
>>>
>>>Nous avons terminé la collecte des données explicatives relatives aux `conditions météorologiques` pour chaque point significatif du point de vue de la production d'énergie solaire, via le jeu de données **POWER Hourly API**.
>>>
>>>Nous enregistrons notre jeu de données actuel pour clore la quatrième partie de notre travail de collecte.

In [16]:
# On enregistre ce dataset contenant l'ensemble des données souhaitées (production + astronomie + météo)
df_cams.to_csv(chemin_datasets_meteo + "raw_2020_2024.csv", index=False)

In [17]:
# Exemple de la manière dont charger ce dataset production+astro+météo final :
df = pd.read_csv(chemin_datasets_meteo + "raw_2020_2024.csv", parse_dates=['datetime_utc'])
display(df.head())
print("Taille du dataset :", df.shape)

/tmp/ipython-input-1806849781.py:2: DtypeWarning: Columns (6,7) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(chemin_datasets_meteo + "raw_2020_2024.csv", parse_dates=['datetime_utc'])


,datetime_utc,Périmètre,Nature,Consommation,Solaire,Ech. physiques,Stockage batterie,Déstockage batterie,TCO Solaire (%),TCH Solaire (%),...,SVT_Nebulosite,SVT_Humidite,BRA_Temperature,BRA_Vitesse_Vent,BRA_Nebulosite,BRA_Humidite,EYG_Temperature,EYG_Vitesse_Vent,EYG_Nebulosite,EYG_Humidite
0,2019-12-31 23:00:00+00:00,PACA,Données définitives,6123.0,0.0,3332.0,-,-,0.0,0.0,...,29.440,57.600,3.230,1.620,5.700,86.880,2.720,1.220,51.750,100.000
1,2019-12-31 23:30:00+00:00,PACA,Données définitives,5907.0,0.0,2837.0,-,-,0.0,0.0,...,24.785,56.165,3.345,1.565,4.635,84.455,2.760,1.405,54.215,98.790
2,2020-01-01 00:00:00+00:00,PACA,Données définitives,5724.0,0.0,2668.0,-,-,0.0,0.0,...,20.130,54.730,3.460,1.510,3.570,82.030,2.800,1.590,56.680,97.580
3,2020-01-01 00:30:00+00:00,PACA,Données définitives,5749.0,0.0,2754.0,-,-,0.0,0.0,...,10.340,54.250,3.650,1.435,3.040,79.380,2.675,1.635,33.325,96.065
4,2020-01-01 01:00:00+00:00,PACA,Données définitives,5700.0,0.0,2886.0,-,-,0.0,0.0,...,0.550,53.770,3.840,1.360,2.510,76.730,2.550,1.680,9.970,94.550


Taille du dataset : (87696, 90)


In [19]:
display(df.iloc[23:28,60:70])

,EYG_TOA,EYG_Clear sky GHI,EYG_Clear sky BHI,EYG_Clear sky DHI,EYG_Clear sky BNI,EYG_GHI,EYG_BHI,EYG_DHI,EYG_BNI,EYG_Reliability
23,253.2484,181.5259,150.7824,30.7435,419.0294,181.0210,149.8874,31.1336,416.6206,1.0
24,268.6971,194.6732,162.9588,31.7144,426.8785,173.7657,126.2110,47.5547,330.7245,1.0
25,276.2950,201.1038,168.8604,32.2434,430.1928,176.9042,126.1173,50.7869,321.3643,1.0
26,275.9122,200.6609,168.3135,32.3474,429.3926,177.5202,127.8147,49.7055,326.1265,1.0
27,267.5549,193.3875,161.3886,31.9990,424.5634,173.1829,126.9076,46.2753,333.8453,1.0
